In [26]:
from astroquery.gaia import Gaia
import matplotlib.pyplot as plt
from os.path import isfile
from astropy.table import Table

# Plot-Formatierung
plt.rcParams['font.size'] = 24.0
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelsize'] = 'medium'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['lines.linewidth'] = 2.0

### Skip the following if you dont want to query the database yourself

In [27]:
Gaia.login() # You might want to create your own account if you want to query the data yourself

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


#### Printing Information about available tables and table contents

In [28]:
def print_available_gaiadr3_tables():    
    tables = Gaia.load_tables(only_names=True)
    for table in tables:
        name: str = table.get_qualified_name()
        if name.startswith("gaiadr3"):
            print(name)
print_available_gaiadr3_tables()

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
gaiadr3.gaiadr3.gaia_source
gaiadr3.gaiadr3.gaia_source_lite
gaiadr3.gaiadr3.astrophysical_parameters
gaiadr3.gaiadr3.astrophysical_parameters_supp
gaiadr3.gaiadr3.oa_neuron_information
gaiadr3.gaiadr3.oa_neuron_xp_spectra
gaiadr3.gaiadr3.total_galactic_extinction_map
gaiadr3.gaiadr3.total_galactic_extinction_map_opt
gaiadr3.gaiadr3.commanded_scan_law
gaiadr3.gaiadr3.allwise_best_neighbour
gaiadr3.gaiadr3.allwise_neighbourhood
gaiadr3.gaiadr3.apassdr9_best_neighbour
gaiadr3.gaiadr3.apassdr9_join
gaiadr3.gaiadr3.apassdr9_neighbourhood
gaiadr3.gaiadr3.dr2_neighbourhood
gaiadr3.gaiadr3.gsc23_best_neighbour
gaiadr3.gaiadr3.gsc23_join
gaiadr3.gaiadr3.gsc23_neighbourhood
gaiadr3.gaiadr3.hipparcos2_best_neighbour
gaiadr3.gaiadr3.hipparcos2_neighbourhood
gaiadr3.gaiadr3.panstarrs1_best_neighbour
gaiadr3.gaiadr3.panstarrs1_join
gaiadr3.gaiadr3.pansta

In [29]:
def print_columns(table = "gaiadr3.gaia_source"):
  gaiadr3_table = Gaia.load_table(table)
  print(f"{'NAME':<35}{'UNIT':<20}{'DESCRIPTION':<200}")
  for column in gaiadr3_table.columns:
    print(f"{str(column.name):<35}{str(column.unit):<20}{str(column.description):<200}")
print_columns()

NAME                               UNIT                DESCRIPTION                                                                                                                                                                                             
solution_id                        None                Solution Identifier                                                                                                                                                                                     
designation                        None                Unique source designation (unique across all Data Releases)                                                                                                                                             
source_id                          None                Unique source identifier (unique within a particular Data Release)                                                                                                               

### Downloading Gaia Data

In [36]:
# TAP-query for downloading data
RERUN = True
filename = "C:/Users/Ruben/Desktop/All_candidates_white_dwarfs.vot"
if (not isfile(filename)) or RERUN:
    query = """
    SELECT 
        l, 
        b, 
        ra,
        ra_error,
        dec,
        dec_error,
        parallax, 
        parallax_error,
        pmra, 
        pmra_error,
        pmdec, 
        pmdec_error,
        phot_g_mean_mag, 
        teff_gspphot,
        bp_rp,
        radial_velocity,
        radial_velocity_error
    FROM 
        gaiadr3.gaia_source AS gdr3
    WHERE 
        teff_gspphot BETWEEN 3500 AND 50000
        AND parallax > 5
        AND phot_g_mean_mag - 5*log10(1000/parallax) + 5 > 6 
    """
    #AND parallax_over_error > 5 -- Ensures useful parallax measurement
    #teff_gspphot BETWEEN 4000 AND 5000

    # Perform TAP query
    job = Gaia.launch_job_async(query)  # This runs for <30min
    results = job.get_results()

    # Print the first few rows
    # print(results)

    # Save to file
    results.write(filename, format="votable", overwrite=True)
    print("Data download complete.")
else:
    print("Data file found. Skipping Gaia query.")

INFO: Query finished. [astroquery.utils.tap.core]
Data download complete.


### mit T und M sollten nur K-dwarfs übrig bleiben vgl. Hertzsprung-Russel

### End of Gaia query part

#### Loading Data-Table

In [7]:
# tab = Table.read("C:/Users/Ruben/Desktop/All_candidates_200pc.vot")
# print(len(tab)) #632951
tab = Table.read("C:/Users/Ruben/Desktop/All_candidates_200pc_noparallaxovererror.vot")
# print(len(tab)) #644675